In [0]:
%pip install python-dotenv pymssql --quiet
dbutils.library.restartPython()

In [0]:
# Realizando as Conexões
import os
from dotenv import load_dotenv

load_dotenv("/Workspace/Users/ik.kukoo@gmail.com/.env", override=True)

STORAGE_ACCOUNT = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

PATH_ESTOQUE    = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data/food_estoque_lojas.csv"
PATH_AVALIACOES = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/batch-data/food_avaliacoes_produto.csv"

SQL_HOST = os.getenv("SQL_HOST")
SQL_DB   = os.getenv("SQL_DATABASE")
SQL_USER = os.getenv("SQL_USERNAME")
SQL_PASS = os.getenv("SQL_PASSWORD")

jdbc_url = (
    f"jdbc:sqlserver://{SQL_HOST}:1433;"
    f"database={SQL_DB};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

jdbc_props = {
    "user":     SQL_USER,
    "password": SQL_PASS,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

print("✅ Config OK")
print(f"   JDBC → {SQL_HOST} / {SQL_DB}")

In [0]:
# Testar Conexão com o SQL Server
# Smoke Test: lê uma tabela de sistema para validar permissão
test_query = "(SELECT GETDATE() AS agora, SYSTEM_USER AS usuario) AS t"

try:
    df_test = spark.read.jdbc(
        url=jdbc_url,
        table=test_query,
        properties=jdbc_props
    )
    df_test.show()
    print("✅ Conexão com SQL Server estabelecida com sucesso!")
except Exception as e:
    print(f"❌ Falha na conexão: {e}")

In [0]:
# Ler os Dados da ADLS
df_estoque = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATH_ESTOQUE)
)

df_avaliacoes = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATH_AVALIACOES)
)

print(f"✅ Estoque:    {df_estoque.count()} linhas")
print(f"✅ Avaliações: {df_avaliacoes.count()} linhas")

In [0]:
# ── Validação de Permissões JDBC — CREATE, INSERT e SELECT ──────────────────
# Usa o conector nativo "sqlserver" do Databricks Serverless
# (mais compatível que o spark.write.jdbc genérico neste ambiente)

# Colunas da tabela — garante a ordem correta na inserção
colunas_tabela = ["id_loja", "sku", "id_lote",
                  "quantidade_disponivel", "estoque_minimo", "dt_snapshot"]

# Opções de conexão com o SQL Server
sqlserver_opts = {
    "host":                   SQL_HOST,
    "port":                   "1433",
    "database":               SQL_DB,
    "user":                   SQL_USER,
    "password":               SQL_PASS,
    "encrypt":                "true",
    "trustServerCertificate": "false",
}

# Teste 1: CREATE + INSERT
# mode="append" cria a tabela automaticamente se não existir e insere os dados
# Uma única chamada já prova permissão de escrita no schema squad3
df_estoque.limit(10).select(*colunas_tabela).write \
    .format("sqlserver") \
    .options(**sqlserver_opts) \
    .option("dbtable", "squad3.food_estoque_lojas") \
    .mode("append") \
    .save()
print("✅ CREATE + INSERT — tabela criada e 10 linhas inseridas com sucesso")

# Teste 2: SELECT
# Lê a tabela de volta para o Spark e confirma que os dados foram persistidos
df_verificacao = spark.read.jdbc(
    url=jdbc_url,
    table="squad3.food_estoque_lojas",
    properties=jdbc_props
)
total = df_verificacao.count()
print(f"✅ SELECT — {total} registros confirmados no SQL Server")
df_verificacao.show(10, truncate=False)

# Resultado final
print("\n🎉 Fluxo ponta a ponta validado com sucesso!")
print("   ADLS Gen2 → Databricks Serverless → Azure SQL Server")
print(f"   Tabela: squad3.food_estoque_lojas | Registros: {total}")

In [0]:
# ── Validação de Permissões JDBC — food_avaliacoes_produto ──────────────────
# Replica o mesmo fluxo validado para a segunda tabela do Squad 3

# Colunas da tabela — garante a ordem correta na inserção
colunas_avaliacoes = ["id_avaliacao", "id_pedido", "id_cliente",
                      "sku", "nota", "dt_avaliacao", "verificada"]

# Teste 1: CREATE + INSERT
df_avaliacoes.limit(10).select(*colunas_avaliacoes).write \
    .format("sqlserver") \
    .options(**sqlserver_opts) \
    .option("dbtable", "squad3.food_avaliacoes_produto") \
    .mode("append") \
    .save()
print("✅ CREATE + INSERT — tabela criada e 10 linhas inseridas com sucesso")

# Teste 2: SELECT
df_verificacao_av = spark.read.jdbc(
    url=jdbc_url,
    table="squad3.food_avaliacoes_produto",
    properties=jdbc_props
)
total_av = df_verificacao_av.count()
print(f"✅ SELECT — {total_av} registros confirmados no SQL Server")
df_verificacao_av.show(10, truncate=False)

# Resultado final
print("\n🎉 Fluxo ponta a ponta validado com sucesso!")
print("   ADLS Gen2 → Databricks Serverless → Azure SQL Server")
print(f"   Tabela: squad3.food_avaliacoes_produto | Registros: {total_av}")